# The sneakers classification problem -- supervised learning

---

## 1. Introduction

This notebook trains and evaluate supervised models to solve the "Popular Sneakers Classification" task.


The following code block contains the main parameters for this notebook.

In [1]:
# Data directory
data_dir = "./data"

batch_size = 128

num_workers = 8

# Model parameters
samples_per_class_list = [None]#[1, 8, 32, 150] # 150 => all training samples
n_versions = 1
max_steps = 120 * 30
max_epochs = 20
add_from_scratch_models = True
add_ImgNet_pretrained_models = False

---
## 2. Setting up the dataset

We will use the Sneakers data module to automatically download and handle the dataset.

In [2]:
from ptbxl_dataset import PTBXLDataModule

datamodule = PTBXLDataModule(data_dir=data_dir, batch_size=batch_size, num_workers=num_workers)

class_names = datamodule.full_dataset.classes

print(datamodule)

{Train dataloader: size=13025}
{Validation dataloader: size=1648}
{Test dataloader: size=1662}
{Predict dataloader: None}


---
## 3. Setting up the models



Let's start by writing the code to support the creation of the backbone, the prediction head, and the supervised model itself.

In [3]:
import torch
from torchvision.models import resnet18
from minerva.models.nets.base import SimpleSupervisedModel
from torchmetrics import Accuracy

from resnet_18_1d import ResNet1D

def generate_backbone(weights=None):
    backbone = ResNet1D(weights, 12, 5)
    backbone.fc = torch.nn.Identity()
    return backbone

def generate_pred_head(backbone_out_dim=512, hidden_dim=512):
    return torch.nn.Sequential(
        torch.nn.Linear(backbone_out_dim, hidden_dim),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden_dim, len(class_names))
    )

# Build a simple supervised model for STL10 using a given backbone
def build_SimpleSupervisedModel(backbone):
  return SimpleSupervisedModel(
    backbone=backbone,
    fc=generate_pred_head(),
    loss_fn=torch.nn.CrossEntropyLoss(),
    train_metrics={"accuracy": Accuracy("multiclass", num_classes=len(class_names))},
    val_metrics  ={"accuracy": Accuracy("multiclass", num_classes=len(class_names))},
    test_metrics ={"accuracy": Accuracy("multiclass", num_classes=len(class_names))},
  )

Let's also create a transform pipeline to generate modified training samples.

In [4]:
import torch
from torchvision.transforms.v2 import Compose, ToImage, ToDtype, Normalize, RandomResizedCrop, RandomHorizontalFlip, ColorJitter, Lambda

precomputed_dataset_stats = {
            'mean': torch.tensor([-0.0019, -0.0015,  0.0005,  0.0017, -0.0011, -0.0005,  0.0003, -0.0010,-0.0017, -0.0021, -0.0012, -0.0011]),
            'std': torch.tensor([0.1699, 0.1573, 0.1645, 0.1416, 0.1476, 0.1366, 0.2178, 0.3177, 0.3082, 0.2941, 0.2830, 0.2346])}

# Set the training set image transformation pipeline
train_transform_pipeline = Compose([Lambda(lambda x: torch.from_numpy(x)),
                                    ToDtype(torch.float32, scale=True),
                                    Lambda(lambda x: (x - precomputed_dataset_stats["mean"].unsqueeze(1)) / precomputed_dataset_stats["std"].unsqueeze(1))
                                   ])

Now, let's create models with different configurations (e.g., initial parameters), and models to be trained with different number of samples per class.

In [5]:
models = {}

from torchvision.models import ResNet50_Weights
import lightning

# Let's set the seeds for reproducibility
lightning.seed_everything(1969)

for version in range(n_versions):

    for train_transform_id, train_transform_pipeline in [ ("aug", train_transform_pipeline), ("notr", None) ]:

        for samples_per_class in samples_per_class_list:

            # -- Add the from scratch model --
            if add_from_scratch_models:
                backbone = generate_backbone()
                models[f"From_Scratch-{train_transform_id}/{samples_per_class}_spc/{max_steps}_steps/v_{version}"] = {
                    "backbone": backbone,
                    "model": build_SimpleSupervisedModel(backbone),
                    "max_steps": max_steps, 
                    "max_epochs": max_epochs, 
                    "samples per class": samples_per_class,
                    "train_transform": train_transform_pipeline,
                    "version": version
                }

            # -- Add the pretrained model: ImageNet weights --
            if add_ImgNet_pretrained_models:
                backbone = generate_backbone(weights=ResNet50_Weights.DEFAULT)
                models[f"Pretrained_ImageNet-{train_transform_id}/{samples_per_class}_spc/{max_steps}_steps/v_{version}"] = {
                    "backbone": backbone,
                    "model": build_SimpleSupervisedModel(backbone),
                    "max_steps": max_steps,
                    "max_epochs": max_epochs, 
                    "samples per class": samples_per_class,
                    "train_transform": train_transform_pipeline,
                    "version": version
                }

print("== The following models were included ==")
for i, k in enumerate(models.keys()):
    print(f"{i:3d} {k}")

Seed set to 1969


== The following models were included ==
  0 From_Scratch-aug/None_spc/3600_steps/v_0
  1 From_Scratch-notr/None_spc/3600_steps/v_0


--- 

## 4. Training the models

In [6]:
from lightning import Trainer
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

# Register stats
from timeit import default_timer as timer
n_configs = len(models)
n_configs_trained = 0
start_time = timer()

for model_name, model_info in models.items():
    print("***********************************")
    print(f" Training model {model_name}")
    print("***********************************")
    loggers = [TensorBoardLogger(save_dir=f"logs/PTBXLDataset/Downstream/", name=model_name),
               CSVLogger(save_dir=f"logs/PTBXLDataset/Downstream/", name=model_name)]
    checkpoint_callback = ModelCheckpoint(monitor="val_loss", mode="min")
    trainer = Trainer(max_epochs=model_info["max_epochs"],
                      # max_steps=model_info["max_steps"], 
                      benchmark=True, 
                      log_every_n_steps=10, logger=loggers,
                      callbacks=[checkpoint_callback])
    
    # Train the model
    trainer.fit(model_info["model"], 
                train_dataloaders=datamodule.train_dataloader(samples_per_class=model_info["samples per class"], 
                                                              transform=model_info["train_transform"]),
                val_dataloaders=datamodule.val_dataloader())

    # Load parameters from best epoch
    print(f"Loading weights from {checkpoint_callback.best_model_path}")
    best_model = SimpleSupervisedModel.load_from_checkpoint(checkpoint_callback.best_model_path,
                                                            backbone=model_info["model"].backbone,
                                                            fc=model_info["model"].fc,
                                                            loss_fn=torch.nn.CrossEntropyLoss(),
                                                            train_metrics={"accuracy": Accuracy("multiclass", num_classes=len(class_names))},
                                                            val_metrics  ={"accuracy": Accuracy("multiclass", num_classes=len(class_names))},
                                                            test_metrics ={"accuracy": Accuracy("multiclass", num_classes=len(class_names))})

    # Test the model
    trainer.test(best_model, dataloaders=datamodule.test_dataloader())

    # Compute and display training statistics
    elapsed = timer() - start_time
    n_configs_trained += 1
    avg = elapsed / n_configs_trained  
    print("-----------------------------------")
    print(f"Training stats")
    print(f"  - Avg time to train models: {avg:.2f} seconds ")
    est_total = avg * n_configs
    est_remaining = est_total - elapsed
    print(f"  - Total # models  : {n_configs} model(s)")
    print(f"  - Models trained  : {n_configs_trained} model(s) in {elapsed:.2f} seconds")
    print(f"  - Remaining models: {n_configs-n_configs_trained} model(s). {est_remaining} s remaining (Estimative)")
    print(f"  - Total time      : {est_total} seconds (estimate: avg * # models)")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type             | Params | Mode 
------------------------------------------------------
0 | backbone | ResNet1D         | 3.8 M  | train
1 | fc       | Sequential       | 265 K  | train
2 | loss_fn  | CrossEntropyLoss | 0      | train
------------------------------------------------------
4.1 M     Trainable params
0         Non-trainable params
4.1 M     Total params
16.456    Total estimated model params size (MB)
78        Modules in train mode
0         Modules in eval mode


***********************************
 Training model From_Scratch-aug/None_spc/3600_steps/v_0
***********************************


Sanity Checking: |                                                   | 0/? [00:00<?, ?it/s]

Training: |                                                          | 0/? [00:00<?, ?it/s]

Validation: |                                                        | 0/? [00:00<?, ?it/s]

...

`Trainer.fit` stopped: `max_epochs=20` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Loading weights from logs/PTBXLDataset/Downstream/From_Scratch-notr/None_spc/3600_steps/v_0/version_0/checkpoints/epoch=12-step=1326.ckpt


Testing: |                                                           | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.8062575459480286     │
│         test_loss         │    0.5488626956939697     │
└───────────────────────────┴───────────────────────────┘

-----------------------------------
Training stats
  - Avg time to train models: 948.22 seconds 
  - Total # models  : 2 model(s)
  - Models trained  : 2 model(s) in 1896.44 seconds
  - Remaining models: 0 model(s). 0.0 s remaining (Estimative)
  - Total time      : 1896.44259212201 seconds (estimate: avg * # models)


--- 
## 5. Plotting the results

In [7]:
import matplotlib.pyplot as plt
import csv

def parse_metrics(metrics_csv_filename):
    metrics = None
    with open(metrics_csv_filename, newline='') as csvfile:
        csv_reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for row in csv_reader:
            if not metrics:
                metrics = { m:[] for m in row }
            else:
                for i, (k,v) in enumerate(metrics.items()):
                    if row[i] != "":
                        if i >= 2:
                            # Add epoch, step, metric_info
                            v.append( (int(row[0]), int(row[1]), float(row[i])) )
                        else:
                            # Add epoch or step info
                            v.append( int(row[i]) )
    return metrics

def plot_results(sorted_list, stats):
    color_array = [ 'b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']
    N = len(sorted_list)
    fig, ax = plt.subplots(1, N, figsize=(N*5, 7), sharey=True)
    ax[0].set_ylabel('Accuracy')
    for i, basename in enumerate(sorted_list):
        ax[i].set_xlabel("Steps")
        values = stats[basename]
        for j, (version, metrics) in enumerate(values.items()):
            val_stats = metrics["val_accuracy"]
            epochs, steps, val_accs = zip(*val_stats)
            ax[i].plot(steps, val_accs, label=f"Val: {version}", color=color_array[j])
            if "test_accuracy" in metrics:
                test_stats = metrics["test_accuracy"]
                ax[i].axhline(y=test_stats[-1][2], color=color_array[j], linestyle='--', label=f"Test: {version}")
        ax[i].set_title(f"{basename}")
        ax[i].grid()
        ax[i].legend()
    plt.show()

In [8]:
import glob

# Parse metrics files
metrics_files = sorted(glob.glob(f"logs/SneakersDataset/Downstream/*/*/2400_steps/v_*/*/metrics.csv"))
stats = {}
for i, f in enumerate(metrics_files):
    #print(f" {i:3d} {f}")
    filename_info = f.split("/")
    basename = filename_info[3] + "-" + filename_info[4]+ "-" + filename_info[5]
    version = filename_info[6]+ "-" + filename_info[7]
    if not basename in stats:
        stats[basename] = { version: parse_metrics(f) }
    else:
        stats[basename][version] = parse_metrics(f)

sorted_list = []
for basename, values in stats.items():
    maxv = 0
    for version, metrics in values.items():
        val_stats = metrics["val_accuracy"]
        if val_stats[-1][2] > maxv: maxv = val_stats[-1][2]
    sorted_list.append((maxv, basename))

sorted_list = [ basename for v, basename in sorted(sorted_list, reverse=True) ]

In [9]:

plot_results(sorted_list, stats)

plot_results(sorted_list=["Pretrained_ImageNet-150_spc-2400_steps", 
                          "Pretrained_ImageNet-32_spc-2400_steps", 
                          "Pretrained_ImageNet-8_spc-2400_steps",
                          "Pretrained_ImageNet-1_spc-2400_steps"], stats=stats)

plot_results(sorted_list=["Pretrained_ImageNet-aug-150_spc-2400_steps", 
                          "Pretrained_ImageNet-aug-32_spc-2400_steps", 
                          "Pretrained_ImageNet-aug-8_spc-2400_steps",
                          "Pretrained_ImageNet-aug-1_spc-2400_steps"], stats=stats)
                          

ValueError: Number of columns must be a positive integer, not 0

<Figure size 0x700 with 0 Axes>